# Foundations · Containers and Azure Container Apps

A **container image** is your program plus everything it needs to run (Python, packages, files), frozen into one artifact. A **container** is a running copy of an image. A **registry** stores images. The same image runs on your laptop, in a teammate's, and in Azure, which removes "it works on my machine".

**Azure Container Apps** runs your image for you: it gives it an HTTPS address, restarts it if it crashes, adds copies under load and removes them when idle.

| Concept | In our setup |
|---|---|
| Image | `acr….azurecr.io/app:<tag>`, built from `fy26/Dockerfile` |
| Registry | Your Azure Container Registry |
| Environment | `cae-…`: the shared network and logging boundary for apps |
| Container App | `ca-…`: your service, with its URL |
| Revision | An immutable snapshot of the app (image + settings). Every deploy creates one |
| Replica | A running copy of a revision. We allow 0 to 2 |
| Ingress | The public HTTPS endpoint, forwarding to port 8000 |
| Probes | Liveness `/health`, readiness `/ready` |

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()
out = course.outputs()
APP, RG, ACR = out["container_app_name"], out["resource_group_name"], out["acr_name"]

## 1 · The Dockerfile

Two stages. The first installs dependencies with `uv` from the lockfile; the second copies only the result and the code, and runs as a non-root user. Smaller image, fewer things to attack, faster starts.

In [ ]:
print((course.FY26 / "Dockerfile").read_text())
print("---- .dockerignore (what is sent to the build) ----")
print((course.FY26 / ".dockerignore").read_text())

## 2 · Images in your registry

`az acr build` sends the build context to Azure and builds there, so you never need Docker on your laptop. Each deploy pushes a new tag; old tags stay, which is how you roll back.

In [ ]:
course.run(["az", "acr", "repository", "show-tags", "--name", ACR, "--repository", "app", "--orderby", "time_desc", "-o", "table"])

## 3 · Revisions and replicas

The active revision serves the traffic. With `min_replicas = 0`, the replica list is empty while nobody calls the service.

In [ ]:
course.run(["az", "containerapp", "revision", "list", "-n", APP, "-g", RG, "--query", "[].{revision:name, active:properties.active, created:properties.createdTime, health:properties.healthState, replicas:properties.replicas}", "-o", "table"])

## 4 · Cold start, measured

Call `/health` twice. If the app had scaled to zero, the first call waits for a replica to start (pull image, start Python, build the handbook index). The second call is fast.

In [ ]:
import time

import httpx2

for attempt in (1, 2):
    start = time.perf_counter()
    status = httpx2.get(f"{course.app_url()}/health", timeout=120).status_code
    print(f"call {attempt}: {status} in {time.perf_counter() - start:.1f} s")

## 5 · Logs

What your service prints goes to the console log. Each request line carries its `request_id`, which also comes back in the `X-Request-ID` response header: the thread that ties a user's complaint to a log line.

In [ ]:
course.run(["az", "containerapp", "logs", "show", "-n", APP, "-g", RG, "--tail", "20", "--format", "text"])

## What to remember

- Image = program + dependencies, built once, run anywhere. Tag every build; never reuse a tag.
- A deploy creates a new revision; the old one keeps serving until the new one is ready.
- Scale to zero saves money and costs a slow first request.
- Probes tell the platform when to restart (liveness) and when to send traffic (readiness).

**Learn more:** [What is a container? (Docker)](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/) · [Deploy cloud-native apps using Azure Container Apps (Learn path)](https://learn.microsoft.com/en-us/training/paths/deploy-cloud-native-applications-to-azure-container-apps/) · [Health probes in Container Apps](https://learn.microsoft.com/en-us/azure/container-apps/health-probes) · [Image pull with managed identity](https://learn.microsoft.com/en-us/azure/container-apps/managed-identity-image-pull)